## Aurora_V4 - kWh

#### <span style="color:red; font-weight:bold"> Instructions for Using this Jupyter Notebook:</span>
This Notebook is using end point value minus start point value of kWh to calculate the kWh within one fiscal year.

<span style="color:royalblue">(0) Install Python packages (one-time setup):</span>
- In terminal, run: **pip3 install scikit-learn**

<span style="color:royalblue">(1) Place **data_clean.py** and this Notebook in the same folder (already done)</span>

<span style="color:royalblue">(2) Place **aurora_v4.meter_info.csv,** **building_complex_master_sheet_fyXX.xlsx**, **special_meters.xlsx**, and the **raw data files** in the same directory (already done)</span> 
- **aurora_v4.meter_info.csv** is exported from the database
- **NOTE:** Filename of the raw data files should include the variable name (i.e., kwh)

<span style="color:royalblue">(3) Modify the **Parameters in Section 1** as needed before running this Notebook:</span> 
- Time range, FY, etc.

<span style="color:royalblue">(4) Run the Notebook:</span> 
- Set **Checked = False** and run it once.
- Check **special_meters_plots_fyXX.pdf** located in **output_dir**.
- If there are any unusual periods in the meter reading data, record them in **special_meters.xlsx** located in **input_dir**.
- If no problem, set **Checked = True** and run the notebook again.

<span style="color:royalblue">**Use R² to define Special Meters (R² < 0.9):**</span> 

- **R² (Coefficient of Determination) Definition:** Measures how closely the meter’s kWh readings follow a perfect linear increase (straight) line over time using linear regression.

- **R² Values and Meaning:**
    - R² ≥ 0.9: Excellent — the meter closely follows the expected linear trend.

    - 0.5 ≤ R² < 0.9: Moderate — the meter shows noticeable deviations; may have some irregular readings.

    - 0 < R² < 0.5: Poor — the meter data is highly irregular.

    - R² = 0: All values missing — no valid data.

    - R² = -1: Missing points at start or end of the fiscal year; scaling applied if enough points exist (> 5 months).

    - R² = -2: Stuck points at start or end of the fiscal year; scaling applied.
 
    - R² = -3: Meter restarts at least 1 time.

This notebook:
- loading raw data
- finding candidates
- making review overlays
- syncing the master sheet
- applying corrections
- exporting final outputs

In [26]:
# TODO: check if removed special is duplicating 
# it overwrites file not appending

# TODO: master csv is only based on current input, no official (separate csv)

# TODO: add restarts to overlays

# TODO: add absoprtion of candidate intervals to larger ones when rerun code with longer date range
# idea: use a run specific candidate filename

# TODO: figure out why / where the restarts are and why they arent at 0 or if they get fixed

In [27]:
# TODO: create separate modules folder

### 1. Parameters

In [28]:
############ CHANGE PARAMETERS AS NEEDED #############

Checked = False  # Set to True after you review the special_meters plot.
Insert = False   # Leave False for now unless you want to push results into the master sheet.
# Insert = False   # Set to False if there's no building_complex_master_sheet_fyXX.xlsx for that fiscal year.

# TODO: make easier selecting time range
# Time Range: Select One Fiscal Year
start_time = "2025-07-23 09:40:50" #"2025-07-23 09:40:50" #"2024-07-01 00:00:00"  #2025-07-23 09:40:50
end_time = "2026-05-08 11:45:00" #"2025-10-17 11:39:02" #"2025-07-01 00:00:00"
# requested_start_time = "2025-07-23 09:40:50"
# requested_end_time = "2025-10-17 11:39:02"

FY = "" #"_fy25"

######################################################

In [29]:
# TODO:
# for finding min and max datetime in the data file
# import pandas as pd

# df = pd.read_csv(var_file, usecols=["datetime"])
# df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

# min_dt = df["datetime"].min()
# max_dt = df["datetime"].max()

# print("min datetime:", min_dt)
# print("max datetime:", max_dt)

In [30]:

# Data Directories
input_dir = "../data/extracts/"  # directory for raw data files & other input files

output_dir = "../data/outputs/"  # directory for data outputs (different from input_dir)

plot_dir = "../data/outputs/plots/"  # directory for plot outputs


# Variable
var = 'kwh'


var_file = input_dir + "harvest_kwh_15min_" + "250723-260508.csv" #"250723-251017.csv" # data file 0, should contain only interval meter readings
meter_info_file = input_dir + "meter_info.csv"  # contains all meter information

# Review/source files
meter_issues_candidates_file = input_dir + "special_meter_candidates.csv"  # auto-generated review file
broken_meters_file = input_dir + "running_list_broken_meters.csv"  # auto-added broken intervals source

# Output Special Meter Files
meter_corrections_file = output_dir + "special_meters_corrections_master_sheet.csv"  # official + auto-broken used for this run
removed_special_meter_data_file = output_dir + "removed_special_meter_data.csv"  # raw data captured inside remove/broken windows

# Output Figures
all_meters_plot = plot_dir + "all_meters_plots" + ".pdf"
review_overlay_plot = plot_dir + "review_special_meters_plots" + ".pdf"

######################################################
if Insert:
    # TODO:
    # commented out for now since about fiscal year sheet
    # insert_sheet = input_dir + "building_complex_master_sheet" + FY + ".xlsx"  # insert output kWh usage into this sheet
    sheet_name = "complex"
    target_col_idx = 10  # insert into column K "Net kWh" (Note: Column A's index = 0)
######################################################

# Annual Output Files
meter_annual_csv = output_dir + "meter_annual_" + var + FY + ".csv"  # annual kwh usage for each meter
building_annual_csv = output_dir + "building_annual_" + var + FY + ".csv"  # annual kwh usage for each building
scaling_detail_csv = output_dir + "meter_scaling_detail" + FY + ".csv"


# Exclude: Meters of buildings equipped with PV and Student Health
meters_with_pv = [
    'bachman_hall_main',
    'campus_ctr_main',
    'dance_bldg_main',
    'gartley_hall_main',
    'warrior_rec_ctr_main'
]
meters_excluded = meters_with_pv + ['student_health_main']  # student_health data is in vitality_v5


# Valid Data Min Length
valid_len = 5*30*96  # A meter should have at least 5-months valid data within 1 fiscal year

# Parameters for data cleaning - no need to change for now
r2_threshold = 0.9

# TODO: figure this out
# If a meter restarts more than 5 times in a fiscal year, treat it as a Special Meter
restarts_thres = 5

# Data Frequency
freq = '15min'

# Schema
schema = 'harvest'
# schema = 'aurora_v4'


### 2. Imports

In [31]:
%load_ext autoreload
%autoreload 2

import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Alignment
import math

import importlib
sys.path.append(os.path.abspath('..'))
import modules.data_clean_TEST as dc # import self-defined module
importlib.reload(dc)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'modules.data_clean_TEST' from 'c:\\Users\\EileenPeppard\\Documents\\GitHub\\harvest_kwh_prep\\modules\\data_clean_TEST.py'>

### 3. Load Data

In [32]:
# Read var and var_rejects tables
raw_df = pd.read_csv(var_file)

raw_df["datetime"] = pd.to_datetime(raw_df["datetime"])
raw_df.head()

,datetime,meter_name,meter_reading
0,2025-07-23 09:45:00,admin_serv_1,1.381510e+06
1,2025-07-23 10:00:00,admin_serv_1,1.381518e+06
2,2025-07-23 10:15:00,admin_serv_1,1.381530e+06
3,2025-07-23 10:30:00,admin_serv_1,1.381541e+06
4,2025-07-23 10:45:00,admin_serv_1,1.381551e+06


### 4. Data Processing

In [33]:
# Sort by meter_name and datetime
raw_df = raw_df.sort_values(by=['meter_name', 'datetime']).reset_index(drop=True)

# Pivot table with every meter as one column
pivoted_df = raw_df.pivot(index='datetime', columns='meter_name', values='meter_reading').reset_index()

# Fill missing timestamps
full_df = dc.fill_missing_timestamps(pivoted_df, freq)

full_df.head()


,datetime,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_engineering_mcc,ag_science_main_1,ag_science_main_2,ag_science_mcc,andrews_amp_main,archtecture_main,...,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,student_health_main,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
0,2025-07-23 09:45:00,1.381510e+06,394675.0000,NaN,NaN,1.137855e+06,8.778624e+06,1.256227e+07,5680.0,325943.0000,...,3.831441e+06,1.022234e+07,3.164514e+07,3.038077e+07,1.752014e+07,122422.0,190848.0000,687433.8816,4.480928e+06,635786.0000
1,2025-07-23 10:00:00,1.381518e+06,394680.0000,NaN,NaN,1.137883e+06,8.778674e+06,1.256230e+07,5680.0,325956.1610,...,3.831449e+06,1.022239e+07,3.164521e+07,3.038084e+07,1.752015e+07,122426.0,190850.2544,687439.0000,4.480942e+06,635790.0000
2,2025-07-23 10:15:00,1.381530e+06,394685.0000,NaN,NaN,1.137913e+06,8.778724e+06,1.256233e+07,5680.0,325970.0966,...,3.831456e+06,1.022243e+07,3.164529e+07,3.038092e+07,1.752016e+07,122430.0,190853.0000,687444.0000,4.480954e+06,635794.0000
3,2025-07-23 10:30:00,1.381541e+06,394689.8851,NaN,NaN,1.137944e+06,8.778775e+06,1.256236e+07,5680.0,325982.8016,...,3.831465e+06,1.022248e+07,3.164537e+07,3.038099e+07,1.752017e+07,122434.0,190856.0000,687449.0000,4.480967e+06,635798.8904
4,2025-07-23 10:45:00,1.381551e+06,394694.6680,NaN,NaN,1.137974e+06,8.778825e+06,1.256239e+07,5680.0,325997.0000,...,3.831473e+06,1.022253e+07,3.164545e+07,3.038106e+07,1.752018e+07,122438.0,190859.0000,687454.0000,4.480981e+06,635803.0000


##### <span style="color:royalblue">Filter Meters and Time Range:</span>

In [34]:
### Filter One: Retain only main meters and filter out sub and PV meters ###

# Step 1: Read meter info
meter_info = pd.read_csv(meter_info_file)

# Step 2: Exclude certain meters first
all_meters = meter_info['meter_name'].unique()
non_exc_meters = [m for m in all_meters if m not in meters_excluded]

# Step 3: Get all 'main' meters from non-PV meters
main_meters_no_exc = meter_info[
    (meter_info['end_use'] == 'main') & 
    (meter_info['meter_name'].isin(non_exc_meters))]['meter_name'].unique()

# Step 4: Filter full_df to keep only non-PV 'main' meters (plus datetime)
columns_to_keep = ['datetime'] + list(full_df.columns.intersection(main_meters_no_exc))
filtered_df = full_df[columns_to_keep]

# Set index as datetime
filtered_df.set_index('datetime', inplace=True)


In [35]:
### Filter Two: Retain only the selected analysis-window data ###

# resolve the requested times to timestamps that actually exist
# in the filled index before slicing and before endpoint calculations.
start_time, end_time = dc.resolve_analysis_window(
    filtered_df.index,
    start_time,
    end_time,
)

print("Resolved start_time:", start_time)
print("Resolved end_time:", end_time)

data = filtered_df.loc[start_time:end_time, :].copy()
data.index = pd.to_datetime(data.index)

# Initial Data Cleaning: Replace all 0s with NaN in the entire DataFrame 
data = data.replace(0, np.nan)

data.head(2)


Resolved start_time: 2025-07-23 09:45:00
Resolved end_time: 2026-05-08 11:45:00


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_science_main_1,ag_science_main_2,andrews_amp_main,archtecture_main,bachman_hall_annex,biomedical_science_main_a,biomedical_science_main_b,...,sherman_main_1,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2025-07-23 09:45:00,1381510.0,394675.0,NaN,1.137855e+06,8.778624e+06,5680.0,325943.000,46789.8008,7.529014e+07,6.013995e+06,...,3.563548e+06,3.831441e+06,1.022234e+07,3.164514e+07,3.038077e+07,1.752014e+07,190848.0000,687433.8816,4.480928e+06,635786.0
2025-07-23 10:00:00,1381518.0,394680.0,NaN,1.137883e+06,8.778674e+06,5680.0,325956.161,46796.0000,7.529032e+07,6.014008e+06,...,3.563556e+06,3.831449e+06,1.022239e+07,3.164521e+07,3.038084e+07,1.752015e+07,190850.2544,687439.0000,4.480942e+06,635790.0


##### <span style="color:royalblue">Sync Meter Corrections Master Sheet:</span>
- Update or build (if not already exists) the master correction workbook of harvest kwh meter readings for this run

NOTE:
- The master corrections sheet is a broader historical/reference log.
- It may contain meters not present in the current raw data for this run.
- Only corrections for meters in the current dataset are actually applied.

In [36]:
# Master sheet combines:
# - reviewed rows already in special_meters_candidates.xlsx
# - approved candidate rows (approved == 1)
# - broken-meter rows from the running broken workbook
master_df = dc.sync_meter_corrections_master_sheet(
    meter_issues_candidates_file,
    broken_meters_file,
    meter_corrections_file,
    start_time,
    end_time,
)

Master correction file saved to ../data/outputs/special_meters_corrections_master_sheet.csv


c:\Users\EileenPeppard\Documents\GitHub\harvest_kwh_prep\modules\data_clean_TEST.py:878: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat([master_df, pd.DataFrame(broken_rows)], ignore_index=True, sort=False)


##### <span style="color:royalblue">Export Removed/Broken Raw Data:</span>
- Save the raw readings that fall inside `remove` / `broken` correction windows before applying corrections.
- Note: overwrites file, not appending

In [37]:
# Export raw data inside remove/broken correction windows before corrections are applied
removed_special_meter_data = dc.export_removed_special_meter_data(
    data,
    meter_corrections_file,
    removed_special_meter_data_file,
)
removed_special_meter_data.head(2)


Removed special meter data exported to ../data/outputs/removed_special_meter_data.csv


,datetime,meter_name,meter_reading,solution,correction_start,correction_end,issue_type/status,description
0,2025-08-21 10:00:00,gilmore_hall_main_b,6.721636e+06,remove,2025-08-19,NaT,broken,
1,2025-08-21 10:15:00,gilmore_hall_main_b,6.721658e+06,remove,2025-08-19,NaT,broken,


##### <span style="color:royalblue">Correct Special Meters:</span>
- If `special_meters_corrections_master_sheet.csv` exists, its reviewed corrections are applied first.
- Also auto-generates `meter_issues_candidates.csv` after special meters are detected.

In [38]:
# Apply corrections from the master sheet
data_corrected = dc.apply_special_meter_corrections(data, meter_corrections_file)
data_corrected.head(2)


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_science_main_1,ag_science_main_2,andrews_amp_main,archtecture_main,bachman_hall_annex,biomedical_science_main_a,biomedical_science_main_b,...,sherman_main_1,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2025-07-23 09:45:00,1381510.0,394675.0,NaN,1.137855e+06,8.778624e+06,5680.0,325943.000,46789.8008,7.529014e+07,6.013995e+06,...,3.563548e+06,3.831441e+06,1.022234e+07,3.164514e+07,3.038077e+07,1.752014e+07,190848.0000,687433.8816,4.480928e+06,635786.0
2025-07-23 10:00:00,1381518.0,394680.0,NaN,1.137883e+06,8.778674e+06,5680.0,325956.161,46796.0000,7.529032e+07,6.014008e+06,...,3.563556e+06,3.831449e+06,1.022239e+07,3.164521e+07,3.038084e+07,1.752015e+07,190850.2544,687439.0000,4.480942e+06,635790.0


##### <span style="color:royalblue">Find Special Meters:</span>

In [39]:
# Find special meters (R² < 0.9) after applying any manualy approved corrections
df_special_meters, df_restarts = dc.find_special_meters(data_corrected, r2_threshold)

##### <span style="color:royalblue">Autocreate special meter candidates:</span>
Update the candidate meter issues workbook for review for this run
- keeps unresolved rows already in the candidate file
- removes rows where approved == 1
- writes newly detected candidate rows from the corrected data (including "special meters")
- excludes active broken meter rows from the candidate workbook

Output:
- prints a summary of detected special meters
- if restart rows exist, creates `special_meter_candidates_restarts.csv`
- creates `special_meter_candidates.csv` with interval candidate rows

In [40]:
# update special meter candidates file for review
# - includes unresolved rows, detected candidate timeframes, and summary derived review rows
df_meter_issues_candidates = dc.update_special_meter_candidates_workbook(
    data_corrected,
    meter_issues_candidates_file,
    broken_meters_file,
    start_time,
    end_time,
    df_bad_meters=df_special_meters,
    df_restarts=df_restarts,
)

df_meter_issues_candidates.head(2)


Restart file saved to ../data/extracts/special_meter_candidates_restarts.csv
Candidate review file saved to ../data/extracts/special_meter_candidates.csv

Special meter summary:
                 meter_name r2                                          info
               admin_serv_1 -3                               restart 1 times
        ag_engineering_main -3                               restart 1 times
         bachman_hall_annex -3                               restart 1 times
          building_037_main -3                               restart 1 times
   building_1171a_to_f_main -3                               restart 1 times
        env_protection_main -3                               restart 2 times
        johnson_hall_a_main -3                               restart 1 times
       kuykendall_hall_main -3                               restart 1 times
            law_school_main -3                               restart 1 times
      maintenance_shop_main -3                      

,meter_name,solution,start_datetime,end_datetime,issue_type,description,suggestion,r2,approved
0,admin_serv_1,,2025-11-02 01:15:00,2025-11-02 01:30:00,restart_or_drop,Negative jump of -1.916399999987334.,review_restart,-3,0
1,ag_engineering_main,,2025-11-02 01:30:00,2025-11-02 01:45:00,restart_or_drop,Negative jump of -1.9677999999839813.,review_restart,-3,0


In [41]:
print("special meters dataframe info:")
print("data_corrected shape:", data_corrected.shape)
print("index type:", type(data_corrected.index))
print("index min:", data_corrected.index.min())
print("index max:", data_corrected.index.max())

special meters dataframe info:
data_corrected shape: (27753, 74)
index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
index min: 2025-07-23 09:45:00
index max: 2026-05-08 11:45:00


##### <span style="color:royalblue">Plot All Special Meters:</span>
Plot review meters with overlay windows
- Red spans  = broken-meter intervals from the broken-meter workbook `running_list_broken_meters.csv`
- Blue spans = candidate intervals from the candidate workbook `special_meter_candidates.csv`
- More blue spans = overlapping candidate intervals
- Purple spans = both red and blue spans


The plot annotation box shows:
- issue_type first
- then R² text if present

blue line = raw meter readings
red spans = broken intervals
blue spans = candidate issue intervals
black dashed lines = restart or drop events

In [42]:
dc.plot_review_meters_with_overlays(
    data,
    meter_issues_candidates_file,
    broken_meters_file,
    review_overlay_plot,
    ylabel=var,
)   


Review overlay plots saved to ../data/outputs/plots/review_special_meters_plots.pdf


##### <span style="color:royalblue">Plot All Meters:</span>
- PDF of all meters (special and non special) after corrections applied by master sheet

In [43]:
dc.plot_all_meters_to_pdf(data_corrected, all_meters_plot, ylabel=var)


All-meter plots saved to ../data/outputs/plots/all_meters_plots.pdf


In [44]:
if not Checked:
    raise SystemExit("Execution stopped because Checked = False")


SystemExit: Execution stopped because Checked = False

c:\Users\EileenPeppard\anaconda3\envs\eileensconda\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


#### <span style="color:red">!!! Note: Check special_meters_plots_fyXX.pdf first before run the following cells !!!</span>

### 5. Calculate (End - Start) Difference for Annual kWh

In [ ]:
### Compute kWh difference between End point and Start point; Scaling applied for some special meters ###

# Step 1: Compute differences
result_df = dc.compute_meter_differences(
    data_corrected,
    start_time,
    end_time,
    df_special_meters,
    df_restarts,
    valid_len=valid_len,
    r2_threshold=r2_threshold,
    restarts_thres=restarts_thres,
)

################ SCALING DETAIL ################
scaling_detail = result_df.reset_index()[[
    "meter_name", "raw_difference", "difference", "R²", "info", "% scaled"
]]

scaling_detail["estimated_kwh"] = (
    scaling_detail["difference"] - scaling_detail["raw_difference"]
)

scaling_detail.to_csv(scaling_detail_csv, index=False)
# scaling_detail.to_csv("../data/outputs/meter_scaling_detail_fy25.csv", index=False)
print("Total estimated kWh:", scaling_detail["estimated_kwh"].sum())
################################################

# export meter-level differences only
df_all = result_df.reset_index()[["meter_name", "difference"]].copy()
df_all.rename(columns={"difference": f"annual_{var}"}, inplace=True)
df_all[f"annual_{var}"] = df_all[f"annual_{var}"].round(1)
df_all.to_csv(meter_annual_csv, index=False)

# skip building-level export for now because there is no meter_info file yet

# # Step 2: Export all meters' differences -> annual kWh usage (CSV, rounded 1 decimal)
# df_all = dc.export_meter_differences(result_df, meter_info_file, meter_annual_csv, var=var)

# # Step 3: Export building-level differences -> annual kWh per building (CSV)
# df_building_sum = dc.export_building_differences(df_all, building_annual_csv, var=var)



##### <span style="color:royalblue">Insert data into master_sheet:</span>

In [ ]:
### Insert kWh difference (annual usage) data into the Master Sheet ####

if Insert:

    # Load workbook and sheet
    wb = load_workbook(insert_sheet)
    ws = wb[sheet_name]

    df_diff = df_building_sum

    # Create mapping: building_complex_name -> annual_{var}
    diff_dict = dict(zip(df_diff['building_complex_name'], df_diff[f'annual_{var}']))

    # Loop rows starting from row 3
    for row in ws.iter_rows(min_row=3):
        building = row[2].value    # Column C (index 2)
        if building in diff_dict:
            val = diff_dict[building]

            # Column K (0-indexed = 10)
            cell = row[target_col_idx]

            # Handle NaN
            if val is None or (isinstance(val, float) and math.isnan(val)):
                cell.value = None
            else:
                cell.value = val

            # Right alignment
            cell.alignment = Alignment(horizontal="right")

    # Save workbook
    wb.save(insert_sheet)
